In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image

# Cihazı belirle
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 📌 Senin oluşturduğun VGG16 modelini tekrar tanımla
class VGG16(nn.Module):
    def __init__(self, num_classes=3):
        super(VGG16, self).__init__()
        
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        
        self.conv5 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.conv6 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        
        self.conv7 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.conv8 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        
        self.conv9 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv10 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        
        self.fc1 = nn.Linear(512 * 7 * 7, 4096)
        self.fc2 = nn.Linear(4096, 4096)
        self.fc3 = nn.Linear(4096, num_classes)

        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.pool(nn.ReLU()(self.conv2(nn.ReLU()(self.conv1(x)))))
        x = self.pool(nn.ReLU()(self.conv4(nn.ReLU()(self.conv3(x)))))
        x = self.pool(nn.ReLU()(self.conv6(nn.ReLU()(self.conv5(x)))))
        x = self.pool(nn.ReLU()(self.conv8(nn.ReLU()(self.conv7(x)))))
        x = self.pool(nn.ReLU()(self.conv10(nn.ReLU()(self.conv9(x)))))

        x = x.view(-1, 512 * 7 * 7)
        x = self.dropout(nn.ReLU()(self.fc1(x)))
        x = self.dropout(nn.ReLU()(self.fc2(x)))
        x = self.fc3(x)
        return x

# 📌 Modeli oluştur ve kaydedilmiş ağırlıkları yükle
model = VGG16(num_classes=3)
model.load_state_dict(torch.load("vgg16_model.pth", map_location=device))  # Modeli yükle
model.to(device)
model.eval()  # Modeli değerlendirme moduna al

print("Model başarıyla yüklendi!")

# 📌 Modeli test etmek için bir görüntü dosyası yükleyelim
def predict_image(image_path):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    image = Image.open(image_path).convert("RGB")  # Görüntüyü aç ve RGB formatına çevir
    image = transform(image).unsqueeze(0).to(device)  # Boyut ekleyerek modele uygun hale getir

    with torch.no_grad():
        outputs = model(image)
        _, predicted = torch.max(outputs, 1)

    class_labels = ["early_blight", "healthy", "late_blight"]  # Sınıf etiketleri
    return class_labels[predicted.item()]

# 📌 Örnek bir test
test_image = "healt2.jpg"  # Test etmek istediğin görseli buraya ekle
result = predict_image(test_image)
print(f"Model tahmini: {result}")
